There is no official standard for SV VCF formats, so we must make some edits to the 1000 Genomes phase3 SV VCF in order to be compatible with Watershed-SV. These edits include:

* Convert the format of CNVs from ALT alleles with varying copy numbers, to a single CN FORMAT field with total (unphased) copy number.
* Rewrite the genotype format to be unphased (e.g. 1|0 -> 0/1).
* Filter for rare variants, as that is what we are primarily interested in
* Create a file of pre-computed SV MAFs.
* Add MODECN and SVLEN fields.

This notebook will take about 5 minutes to run.

## Setup

In [1]:
BiocManager::install('VariantAnnotation')

library(data.table)
library(VariantAnnotation)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org

Bioconductor version 3.21 (BiocManager 1.30.25), R 4.5.0 (2025-04-11)

Installing package(s) 'VariantAnnotation'

also installing the dependency ‘BSgenome’


Old packages: 'alabaster.base', 'AnnotationHub', 'AnVIL', 'base64enc', 'BH',
  'bigrquery', 'BiocFileCache', 'BiocGenerics', 'BiocManager', 'BiocParallel',
  'blob', 'boot', 'broom', 'bslib', 'cli', 'clock', 'cluster', 'colorspace',
  'commonmark', 'cowplot', 'cpp11', 'credentials', 'crosstalk', 'curl',
  'data.table', 'dbplyr', 'DelayedArray', 'DESeq2', 'devtools', 'diffobj',
  'digest', 'downlit', 'dplyr', 'DT', 'dtplyr', 'edgeR', 'evaluate',
  'ExperimentHub', 'fansi', 'fitdistrplus', 'forcats', 'foreign',
  'futile.logger', 'future', 'future.apply', 'gargle', 'generics',
  'GenomeInfoDb', 'gert', 'ggplot2', 'ggridges', 'gh', 'globals'

# CNV format conversion and unphasing
In the 1000G VCF file, CNVs are formatted such that \<CN1\> is the REF allele, with other copy numbers as ALT alleles. For example, for a variant with ALT alleles \<CN0\> and \<CN2\>, an individual with the genotype 0|2 (reference allele | 2nd ALT allele) would have a total copy number of 3 (CN1+CN2). However, Watershed-SV expects a separate FORMAT/CN field in the VCF field for CNVs, which we create here.

Additionally, Watershed-SV cannot yet use phasing information, so we simplify the VCF to be unphased (e.g. 1|0 -> 0/1).

These steps must be done with bcftools and awk for computational efficiency. This streams VCF information line-by-line instead of loading the entire file into memory at once.

In [2]:
# Unphase
system('bcftools +setGT ALL.wgs.mergedSV.v8.20130502.svs.genotypes.vcf.gz -Ob -- -t a -n u > unphased.bcf.gz')

In [3]:
# Calculate total CN for CNVs, then annotate the original VCF with this information.
system('
  bcftools view unphased.bcf.gz -Ou -i \'INFO/SVTYPE=="CNV" || INFO/SVTYPE=="DUP" || INFO/SVTYPE=="DEL"\' | \\
  bcftools query -f \'%CHROM\\t%POS\\t%REF\\t%ALT[\\t%GT]\\n\' | \\
  mawk \'{
    n_alt = split($4, alts, ","); # $4 is the ALT column
    for (i=1; i<=n_alt; i++) {
      match(alts[i], /[0-9]+/) # Extract "#" from "<CN#>"
      alts_cn[i] = substr(alts[i], RSTART, RLENGTH) }
    out = $1 "\\t" $2; # CHROM ($1) and POS ($2) cols to be indexed with tabix
    for (s=5; s<=NF; s++) { # Loop through all the genotype fields, calc total CN
      split($s, gt, /[\\/|]/);
      cn1 = (gt[1]==0 ? 1 : alts_cn[gt[1]]);
      cn2 = (gt[2]==0 ? 1 : alts_cn[gt[2]]);
      out = out "\\t" cn1+cn2; }
    print out;
  }\' > cn_table.tsv
')

system('bgzip -f cn_table.tsv')

system('tabix -s 1 -b 2 -e 2 cn_table.tsv.gz')

writeLines('##FORMAT=<ID=CN,Number=1,Type=Integer,Description="Total copy number">', 'CN_header.txt')
system('
  bcftools annotate unphased.bcf.gz -o unphased_cn.vcf.gz \\
    -a cn_table.tsv.gz \\
    -c CHROM,POS,FORMAT/CN \\
    -h CN_header.txt
')

The remaining steps are more lightweight and can be done more comfortably with R's VariantAnnotation package. 

In [4]:
vcf <- readVcf('unphased_cn.vcf.gz')

# Get MAFs

In [5]:
afs <- sapply(info(vcf)$AF,sum) # Sum AF of all alt alleles
mafs <- fifelse(afs<0.5,afs,1-afs)

Create MAF file for use by Watershed-SV later.

In [6]:
nms <- names(rowRanges(vcf))
fwrite(data.table(SV=nms,af=mafs),'1kGp3_sv_maf_file.tsv', sep='\t')
system('gcloud storage cp 1kGp3_sv_maf_file.tsv $WORKSPACE_BUCKET/data/derived/')

# Subset to rare variants only

In [7]:
vcf <- vcf[mafs<0.01,]

# Convert certain DUP/DELs to CNV
There are variants which are labeled with SVTYPE=DUP/DEL, but are still formatted as if they were SVTYPE=CNV, with <CN#> ALT alleles. They are labelled as DUP if the AF for \<CN0\> is 0, or as DEL if the AF for everything except \<CN0\> is 0. \
A more correct way to do this would be to keep the SVTYPE as DUP/DEL, remove the alt allele that has AF=0, and recode the genotype field appropriately. However, that would be more difficult to implement.

In [8]:
weird_dups_dels <- (info(vcf)$SVTYPE=='DUP' | info(vcf)$SVTYPE=='DEL') & lengths(alt(vcf))>1
info(vcf)$SVTYPE[weird_dups_dels] <- 'CNV'

And here are a few more SVTYPEs that we will classify more broadly as DELs:

In [9]:
special_dels <- (info(vcf)$SVTYPE=='DEL_HERV'  |
                 info(vcf)$SVTYPE=='DEL_ALU'   |
                 info(vcf)$SVTYPE=='DEL_LINE1' |
                 info(vcf)$SVTYPE=='DEL_SVA'   )
info(vcf)$SVTYPE[special_dels] <- 'DEL'

# Create `MODECN` field
R's `mode()` gets a data structure' type, it's not "mode" in the mathematical sense. We have to make our [own function](https://stackoverflow.com/a/8189441). \
Watershed-SV depends on a `MODECN` field being present in the VCF ([see source code](https://github.com/jasonbhn/Watershed-SV/blob/2f53ae75c9723ca88cb2c9424d19982f1067814f/scripts/executable_scripts/sv_utils.py#L128)). \
(This code block may take a little while to run.)

In [10]:
mode <- function(x) {
  u <- unique(x)
  u[which.max(tabulate(match(x,u)))]
}

modecn <- apply(geno(vcf)$CN,1,mode)

## Update VCF

In [11]:
# Obligatory updating of VCF header.
new_info   <- DataFrame(row.names='MODECN', Number='1', Type='Integer', Description='Most frequent copy number (CN)')
info(header(vcf)) <- rbind(info(header(vcf)), new_info  )

# Insert new INFO/MODECN field.
info(vcf)$MODECN <- modecn

# Create SVLEN field
This field already exists in the VCF file, but not all variants have it so we must fill it in.

In [12]:
no_svlen <- lengths(info(vcf)$SVLEN) == 0 # Find variants where the SVLEN field is not already extant
info(vcf)$SVLEN[no_svlen] <- abs( info(vcf)$END - start(vcf) )[no_svlen]

# Write

In [13]:
writeVcf(vcf,   'ALL.wgs.mergedSV.v8.20130502.svs.genotypes-edited.vcf')
system('bgzip -f ALL.wgs.mergedSV.v8.20130502.svs.genotypes-edited.vcf')

In [14]:
system('gcloud storage cp ALL.wgs.mergedSV.v8.20130502.svs.genotypes-edited.vcf.gz $WORKSPACE_BUCKET/data/derived/')